# Nuvoton Industry Project -- Elevator People Counting on the M55M1

This notebook covers the following steps to calibrate and export the YOLO model onto monochrome device specifications:
1. **Conda environment setup** running inside Colab with `condacolab`.
2. **Dataset downloads and layouts** from HuggingFace to a 90/10 layout split.
3. **Local training running** with constant 192x192 monochrome variables triggers inputs.
4. **Evaluation metrics validation** layout scripts.
5. **Onnx/TFLite layout conversion exports** matching Ethos-U55 parameters.

In [1]:
!pip install -q condacolab
import condacolab
condacolab.install_from_url("https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-Linux-x86_64.sh")

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:10
🔁 Restarting kernel...


In [1]:
!conda --version

conda 25.3.1


In [2]:
GIT_REPO_URL = "https://github.com/bencejdanko/m55m1-ElevatorCounting-YOLOv8n"
!git clone {GIT_REPO_URL}

import os
dir_name = os.path.basename(GIT_REPO_URL).replace('.git', '')
%cd {dir_name}/yolov8_ultralytics

Cloning into 'm55m1-ElevatorCounting-YOLOv8n'...
remote: Enumerating objects: 924, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 924 (delta 22), reused 61 (delta 16), pack-reused 845 (from 2)
Receiving objects: 100% (924/924), 148.54 MiB | 33.65 MiB/s, done.
Resolving deltas: 100% (222/222), done.
/content/m55m1-ElevatorCounting-YOLOv8n/yolov8_ultralytics


In [ ]:
!conda create --name yolov8_DG python=3.10 -y

!conda run -n yolov8_DG python -m pip install --upgrade pip setuptools
!conda run -n yolov8_DG python -m pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu118

# Install current local package using .[export]
!conda run -n yolov8_DG python -m pip install .[export]

# Additional supporting packages needed for download/export
!conda run -n yolov8_DG python -m pip install datasets onnx2tf onnx

In [ ]:
# Download bdanko/overhead-person-detection dataset
# more info https://github.com/bencejdanko/m55m1-ElevatorCounting-YOLOv8n
!conda run -n yolov8_DG python download_hf_dataset.py --dataset bdanko/overhead-person-detection --out-dir datasets/overhead_monochrome

In [13]:
!conda run -n yolov8_DG env MPLBACKEND=Agg python dg_train.py \
  --model-cfg relu6-yolov8.yaml \
  --data datasets/overhead_monochrome/data.yaml \
  --imgsz 192 \
  --weights yolov8n.pt \
  --epochs 10 \
  --device 0

WARNING ⚠️ no model scale passed. Assuming scale='n'.
Transferred 355/451 items from pretrained weights
{'cfg': None, 'data': 'datasets/overhead_monochrome/data.yaml', 'epochs': 10, 'batch': 64, 'imgsz': 192, 'device': ['0'], 'optimizer': 'SGD', 'workers': 2, 'project': 'runs/train', 'name': 'exp', 'exist_ok': False, 'patience': 0, 'cache': False, 'close_mosaic': 3, 'resume': False, 'lr0': 0.01, 'lrf': 0.01, 'cos_lr': False, 'scale': 0.5, 'mixup': 0.0, 'copy_paste': 0.0}
New https://pypi.org/project/ultralytics/8.4.24 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.42 🚀 Python-3.10.20 torch-2.5.1+cu118 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=relu6-yolov8.yaml, data=datasets/overhead_monochrome/data.yaml, epochs=10, time=None, patience=0, batch=64, imgsz=192, save=True, save_period=-1, cache=False, device=['0'], workers=2, project=runs/train, name=exp2, exist_ok=False, pretrained=True, optimizer=SGD, verbose=True, seed=0, dete

In [16]:
# validation
# Note the experimental run you saved at!
EXPERIMENT="exp2"
!conda run -n yolov8_DG env MPLBACKEND=Agg python dg_val.py \
  --weights runs/train/{EXPERIMENT}/weights/best.pt \
  --data datasets/overhead_monochrome/data.yaml \
  --imgsz 192 \
  --device 0

{'weights': 'runs/train/exp2/weights/best.pt', 'data': 'datasets/overhead_monochrome/data.yaml', 'annotations': None, 'device': '0', 'imgsz': 192, 'no_separate_outputs': False}
Ultralytics YOLOv8.2.42 🚀 Python-3.10.20 torch-2.5.1+cu118 CUDA:0 (Tesla T4, 14913MiB)
relu6-YOLOv8 summary (fused): 200 layers, 3278259 parameters, 0 gradients
                   all       1345       2868      0.932      0.855      0.935      0.548
WARNING ⚠️ ConfusionMatrix plot failure: No module named 'seaborn'
WARNING ⚠️ ConfusionMatrix plot failure: No module named 'seaborn'
Speed: 0.1ms preprocess, 1.3ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to /content/m55m1-ElevatorCounting-YOLOv8n/runs/detect/val


val: Scanning /content/m55m1-ElevatorCounting-YOLOv8n/yolov8_ultralytics/datasets/overhead_monochrome/val/labels.cache... 1345 images, 55 backgrounds, 0 corrupt: 100%|██████████| 1345/1345 [00:00<?, ?it/s]
val: Scanning /content/m55m1-ElevatorCounting-YOLOv8n/yolov8_ultralytics/dat

In [21]:
!conda run -n yolov8_DG python -m pip install py-cpuinfo

  Using cached py_cpuinfo-9.0.0-py3-none-any.whl.metadata (794 bytes)
Using cached py_cpuinfo-9.0.0-py3-none-any.whl (22 kB)



In [22]:
# graph intermediary to onnx vectors
!conda run -n yolov8_DG env MPLBACKEND=Agg python nu_export_tflite_int8.py --format onnx --weights ./runs/train/{EXPERIMENT}/weights/best.pt --img 192

{'format': 'onnx', 'weights': './runs/train/exp2/weights/best.pt', 'imgsz': 192, 'device': 'cpu', 'img_dir': None, 'n_img': 200, 'out': 'calib_data.npy'}
Ultralytics YOLOv8.2.42 🚀 Python-3.10.20 torch-2.5.1+cu118 CPU (Intel Xeon 2.00GHz)
relu6-YOLOv8 summary (fused): 200 layers, 3278259 parameters, 0 gradients

PyTorch: starting from 'runs/train/exp2/weights/best.pt' with input shape (1, 3, 192, 192) BCHW and output shape(s) ((1, 576, 64), (1, 144, 64), (1, 36, 64), (1, 576, 1), (1, 144, 1), (1, 36, 1)) (6.5 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.31'] not found, attempting AutoUpdate...


requirements: AutoUpdate success ✅ 1.4s, installed 1 package: ['onnxslim>=0.1.31']
requirements: ⚠️ Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.19.1 opset 19...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export success ✅ 2.6s, saved as 'runs/train/exp2/weights/best.onnx' (11.5 MB)

Export complete (4.0s)
Results saved to /content

In [24]:
# graph intermediary to onnx vectors
!conda run -n yolov8_DG env MPLBACKEND=Agg python nu_export_tflite_int8.py --format onnx --weights ./runs/train/{EXPERIMENT}/weights/best.pt --img 192


{'format': 'onnx', 'weights': './runs/train/exp2/weights/best.pt', 'imgsz': 192, 'device': 'cpu', 'img_dir': None, 'n_img': 200, 'out': 'calib_data.npy'}
Ultralytics YOLOv8.2.42 🚀 Python-3.10.20 torch-2.5.1+cu118 CPU (Intel Xeon 2.00GHz)
relu6-YOLOv8 summary (fused): 200 layers, 3278259 parameters, 0 gradients

PyTorch: starting from 'runs/train/exp2/weights/best.pt' with input shape (1, 3, 192, 192) BCHW and output shape(s) ((1, 576, 64), (1, 144, 64), (1, 36, 64), (1, 576, 1), (1, 144, 1), (1, 36, 1)) (6.5 MB)

ONNX: starting export with onnx 1.19.1 opset 19...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export success ✅ 1.2s, saved as 'runs/train/exp2/weights/best.onnx' (11.5 MB)

Export complete (2.5s)
Results saved to /content/m55m1-ElevatorCounting-YOLOv8n/yolov8_ultralytics/runs/train/exp2/weights
Predict:         yolo predict task=detect model=runs/train/exp2/weights/best.onnx imgsz=192  
Validate:        yolo val task=detect model=runs/train/exp2/weights/best.onnx imgsz=192 d

In [25]:
# calibrations referencing saved monochrome folder
!conda run -n yolov8_DG env MPLBACKEND=Agg python generate_calib_data.py \
  --img-size 192 192 \
  --n-img 4 \
  -o calib_data_192_n4_rgb.npy \
  --img-dir datasets/overhead_monochrome/train/images

12103
convert from BGR to RGB format!!
(4, 192, 192, 3)
calib_datas.shape: (4, 192, 192, 3)



In [27]:
# Run onnx2tf to extract TFLite integers graph fully
!rm -rf saved_model
!conda run -n yolov8_DG env MPLBACKEND=Agg onnx2tf -i runs/train/{EXPERIMENT}/weights/best.onnx -oiqt -cind images calib_data_192_n4_rgb.npy "[[[[0,0,0]]]]" "[[[[1,1,1]]]]"


Model optimizing started ============================================================
Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃            ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Add        │ 6              │ 6                │
│ Clip       │ 65             │ 65               │
│ Concat     │ 13             │ 13               │
│ Constant   │ 147            │ 147              │
│ Conv       │ 71             │ 71               │
│ MaxPool    │ 3              │ 3                │
│ Reshape    │ 6              │ 6                │
│ Resize     │ 2              │ 2                │
│ Transpose  │ 6              │ 6                │
│ Model Size │ 11.5MiB        │ 11.5MiB          │
└────────────┴────────────────┴──────────────────┘

Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃            ┃ Original Model ┃ Simplified Model ┃
┡━

In [29]:
!conda run -n yolov8_DG python -m pip install ethos-u-vela

  Using cached ethos_u_vela-5.0.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 29.1 MB/s  0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.12.19
    Uninstalling flatbuffers-25.12.19:
      Successfully uninstalled flatbuffers-25.12.19


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
onnx2tf 1.29.24 requires flatbuffers==25.12.19, but you have flatbuffers 24.3.25 which is incompatible.



In [30]:
!conda run -n yolov8_DG vela saved_model/best_integer_quant.tflite \
    --accelerator-config ethos-u55-128 \
    --output-dir /content/ \
    --optimise Performance



Warning (supported operators) operator:DEQUANTIZE ofm:PartitionedCall:2
Reason: Unsupported opType

Warning (supported operators) operator:QUANTIZE ofm:tfl.quantize
Reason: Operation has tensor with unsupported DataType Float32
Feature-map dataTypes must be one of {"INT16", "INT32", "INT64", "INT8", "UINT8"}

Warning (supported operators) operator:DEQUANTIZE ofm:PartitionedCall:0
Reason: Unsupported opType

Warning (supported operators) operator:DEQUANTIZE ofm:PartitionedCall:3
Reason: Unsupported opType

Warning (supported operators) operator:DEQUANTIZE ofm:PartitionedCall:5
Reason: Unsupported opType

Warning (supported operators) operator:DEQUANTIZE ofm:PartitionedCall:1
Reason: Unsupported opType

Warning (supported operators) operator:DEQUANTIZE ofm:PartitionedCall:4
Reason: Unsupported opType
CPU performance estimation for "Passthrough" not implemented
CPU performance estimation for "Passthrough" not implemented
CPU performance estimation for "Passthrough" not implemented
CPU pe

In [31]:
!ls -lh /content/*_vela.tflite

-rw-r--r-- 1 root root 2.7M Mar 23 05:03 /content/best_integer_quant_vela.tflite
